# Corporate Financial Risk Analysis Assistant

In [1]:
import sys
sys.version

'3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]'

## Importing all Necessary Libraries

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from transformers import pipeline

from neo4j import GraphDatabase
from openai import OpenAI

from typing import TypedDict
from langgraph.graph import StateGraph

import os
import re
import json
import time
import requests
from dotenv import load_dotenv
import shutil
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering

C:\Users\reign\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading the Dataset
- Loading all PDF files and storing them in one single variable

In [3]:
# Loading a single file to verify content

# Loading one PDF first
file_path = "data/apple_10k.pdf"

loader = PyPDFLoader(file_path)

docs = loader.load()

# Printing the number of pages loaded
print(f"Total pages loaded: {len(docs)}")

# Printing the first page content (small preview)
print("\n<-- SAMPLE CONTENT -->\n")
print(docs[0].page_content[:500])

Total pages loaded: 65

<-- SAMPLE CONTENT -->

UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒ ANNUAL  REPORT PURSUANT T O SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 27 , 2025
or
☐ TRANSITION REPORT PURSUANT T O SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from              to             .
Commission File Number: 001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California 94-24


##### Merging all files into one variable

In [4]:
data_path = "data"

all_docs = []

for file in os.listdir(data_path):
    if file.endswith(".pdf"):
        full_path = os.path.join(data_path, file)
        loader = PyPDFLoader(full_path)
        docs = loader.load()

        # Adding source metadata
        for doc in docs:
            doc.metadata["source"] = file
        
        all_docs.extend(docs)

print(f"Total documents loaded: {len(all_docs)}\n")
print(all_docs[0].page_content[:1016]) # Page Number 1

Total documents loaded: 521

UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
___________________________________________
FORM 10-K
___________________________________________
(Mark One)
☒ ANNUAL  REPORT PURSUANT T O SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 31 , 2025
OR
☐ TRANSITION REPORT PURSUANT T O SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from              to             .
Commission file number: 001-37580
___________________________________________
Alphabet Inc.
(Exact name of registrant as specified in its charter)
___________________________________________
Delaware 61-1767919
(State or other jurisdiction of incorporation or organization) (I.R.S. Employer Identification No.)
1600 Amphitheatre Parkway
Mountain V iew, CA 94043
(Address of principal executive offices, including zip code)
(650) 253-0000
(Registrant's telephone number, including area code)
Securitie

In [5]:
print("\n<-- SAMPLE METADATA -->\n")
print(all_docs[7].metadata)


<-- SAMPLE METADATA -->

{'producer': 'Skia/PDF m146', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36', 'creationdate': '2026-04-10T07:57:28+00:00', 'title': 'goog-20251231', 'moddate': '2026-04-10T07:57:28+00:00', 'source': 'alphabet_10K.pdf', 'total_pages': 133, 'page': 7, 'page_label': '8'}


## Text Chunking
- Breaking the complete 521 pages document into small chunks or parts with some overlap

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [7]:
chunks = text_splitter.split_documents(all_docs)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 2189


In [8]:
print("\n<-- SAMPLE CHUNK -->\n")
print(chunks[10].page_content)


<-- SAMPLE CHUNK -->

Item 9A. Controls and Procedures 89
Item 9B. Other Information 89
Item 9C. Disclosure Regarding Foreign Jurisdictions that Prevent Inspections 90
PART III
Item 10. Directors, Executive Officers, and Corporate Governance 91
Item 11. Executive Compensation 91
Item 12. Security Ownership of Certain Beneficial Owners and Management and Related Stockholder
Matters91
Item 13. Certain Relationships and Related Transactions, and Director Independence 91
Item 14. Principal Accountant Fees and Services 91
PART IV
Item 15. Exhibits, Financial Statement Schedules 92
Item 16. Form 10-K Summary 96
Signatures
2.4/10/26, 1:27 PM goog-20251231
https://www.sec.gov/Archives/edgar/data/1652044/000165204426000018/goog-20251231.htm 3/133


In [9]:
# Add chunk_id metadata
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
    
print("\n<-- SAMPLE METADATA -->\n")
print(chunks[10].metadata)


<-- SAMPLE METADATA -->

{'producer': 'Skia/PDF m146', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36', 'creationdate': '2026-04-10T07:57:28+00:00', 'title': 'goog-20251231', 'moddate': '2026-04-10T07:57:28+00:00', 'source': 'alphabet_10K.pdf', 'total_pages': 133, 'page': 2, 'page_label': '3', 'chunk_id': 10}


## Manual Embedding
- Converting the text into vector of numbers manually.
- This part is just for Debugging and Understanding and is not included in the final pipeline.

In [10]:
# Loading a pre-trained model

# embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", local_files_only=True)

In [11]:
# Embedding a single chunk to test the model
sample_text = chunks[0].page_content

s_embedding = embedding_model.encode(sample_text)

print(f"Embedding length: {len(s_embedding)}")
print(s_embedding[:10])  # Printing first 10 values

Embedding length: 384
[-0.07352683 -0.01083049 -0.05216291 -0.00918295 -0.0546556   0.06752949
  0.00618795  0.09189996 -0.05595456  0.01993592]


##### Batch Embedding

In [12]:
# Embedding all the chunks in Batches
texts = [chunk.page_content for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True
)

print(f"Total embeddings created: {len(embeddings)}")
print(f"Shape: {embeddings.shape}")  # Checking Shape

Batches: 100%|██████████| 69/69 [00:55<00:00,  1.23it/s]

Total embeddings created: 2189
Shape: (2189, 384)


In [13]:
print(embeddings[10,:20])  # Printing first 20 values of 10th index position

[-0.05771916  0.03734257  0.04011412 -0.03072656  0.03022867  0.07548136
  0.06720416  0.01532797 -0.04551148 -0.04483572  0.08812208  0.01085629
 -0.05970158 -0.03705528 -0.06378383 -0.06259182  0.03253276 -0.02287522
 -0.06625237  0.00609654]


In [14]:
# Checking how similar the embedding of two chunks are
sim = cosine_similarity(
    [embeddings[0]],
    [embeddings[1]]
)

print(sim)

[[0.3948861]]


## Vector Database using ChromaDB <I>(Automatic Embedding)</I>
- Before we were doing Embedding manually but ChromaDB expects a wrapper function to generate embeddings, store them and retrieve later automatically.
- So, this part replaces the previous manual Embedding part.

In [15]:
embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

C:\Users\reign\AppData\Local\Temp\ipykernel_15224\1084410045.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 718738aa-00fd-405d-828b-241bd7407a3f)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, 

In [16]:
current_dir = os.getcwd()  # or use __file__ if script
# project_root = os.path.abspath(os.path.join(current_dir, "../"))

db_path = os.path.join(current_dir, "chroma_db")

print("Deleting:", db_path)

shutil.rmtree(db_path, ignore_errors=True)

Deleting: C:\Users\reign\Multi-Agent Research Assistant\chroma_db


In [17]:
db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_function,
    persist_directory="chroma_db"
)

In [18]:
print(db._collection.count())

2189


In [20]:
query = "What are the main risks mentioned in the report?"

results = db.similarity_search(query, k=5) # k denotes how many relevant answers to store or display

for i, res in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(res.page_content[:1000])
    print("\nSource:", res.metadata)


--- Result 1 ---
Item 1A.    Risk Factors
The following summarizes factors that could have a material adverse effect on the Compan y’s business, reputation, results of
operations, financial condition and stock price. The Company may not be able to accurately predict, control or mitigate these risks.
Statements in this section are based on the Company’ s beliefs and opinions regarding matters that could materially adversely affect
the Company in the future and are not representations as to whether such matters have or have not occurred previously . The risks
and uncertainties described below are not exhaustive and should not be considered a complet e statement of all potential risks or
uncertainties that the Company faces or may face in the future.
This section should be read in conjunction with Part II, Item 7, “Management’ s Discussion and Analysis of Financial Condition and

Source: {'producer': 'Skia/PDF m146', 'creationdate': '2026-04-10T07:48:50+00:00', 'page_label': '8', 'title'

In [21]:
# Checking Similarity using Cosine Similarity Score (High value means more similar)
results = db.similarity_search_with_score(query, k=5)

for res, score in results:
    print(score)
    print(f"{res.page_content[:100]}\n")

0.8067889213562012
Item 1A.    Risk Factors
The following summarizes factors that could have a material adverse effect 

0.8121867775917053
incorporated by reference into this Annual Report on Form 10-K.
ITEM 1A. RISK FACTORS
You should car

0.9075179696083069
property theft, claims, litigation, regulatory investigations, significant liability, reputational d

0.9085608124732971
risks and uncertainties that could cause our actual results to differ materially from those in the f

0.9110981225967407
Item 7 of Part II, “Management’s Discussion and Analysis of Financial Condition and Results of Opera



## RAG Pipeline

In [22]:
#llm = pipeline(
#    "text2text-generation",
#    model="google/flan-t5-base",   # better than distilgpt2
#    max_length=512
#)

In [23]:
llm = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=512,
    temperature=0.2,      # reduces randomness
    repetition_penalty=1.2,
    do_sample=True
)

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 4c6c20e1-9e4b-4752-afe5-0ed92f9e2ed3)')' thrown while requesting HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 39932949-bcf0-4e09-91ac-8ebd66be4b55)')' thrown while requesting HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 8fc3207d-c622-48a0-a3a4-0396189b6bf9)')' thrown while requesting HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json
Retrying in 4s [Retry 3/5].
D

In [24]:
# Creating Retriever
retriever = db.as_retriever(search_kwargs={"k": 5})

#### RAG without Map-Reduce

In [25]:
def build_prompt(context, question):
    return f""" You are a financial analyst assistant.
                Answer the question using ONLY the context below.

                Extract key points clearly.
                Be precise and structured.
                Answer in clear bullet points.
                
                Do NOT say "I don't know" unless absolutely nothing relevant exists.

                Context:
                {context}

                Question:
                {question}

                Answer:
                """

In [26]:
def rag_query(question):
    docs = retriever.invoke(question)
    
    # context = "\n\n".join([doc.page_content for doc in docs])
    
    context = ""
    for i, doc in enumerate(docs):
        context += f"\n### Excerpt {i+1} ###\n{doc.page_content}\n"

    
    prompt = build_prompt(context, question)
    
    response = llm(prompt)
    
    return response[0]['generated_text'].replace(prompt, ""), docs

In [27]:
question = "What risks does Alphabet face and how do they mitigate them?"

answer, docs = rag_query(question)

print("\n--- ANSWER ---\n")
print(answer)

print("\n--- SOURCES ---\n")
for doc in docs:
    print(doc.metadata)

Token indices sequence length is longer than the specified maximum sequence length for this model (1025 > 512). Running this sequence through the model will result in indexing errors



--- ANSWER ---

Our operations and financial results are subject to various risks and uncertainties, including but not limited to those described below, which could harm our business, reputation, financial condition, and operating results, and may affect the trading price and price volatility of our Class A and Class C stock.

--- SOURCES ---

{'creationdate': '2026-04-10T07:57:28+00:00', 'chunk_id': 49, 'source': 'alphabet_10K.pdf', 'producer': 'Skia/PDF m146', 'total_pages': 133, 'page': 12, 'moddate': '2026-04-10T07:57:28+00:00', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36', 'page_label': '13', 'title': 'goog-20251231'}
{'page_label': '26', 'creationdate': '2026-04-10T07:57:28+00:00', 'title': 'goog-20251231', 'producer': 'Skia/PDF m146', 'source': 'alphabet_10K.pdf', 'page': 25, 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',

#### RAG with Map-Reduce

In [28]:
def map_step(docs, question):
    summaries = []

    for doc in docs:
        prompt = f"""You are a financial analyst.

Extract ONLY the information relevant to the question.

Rules:
- Do NOT repeat the question
- Keep answer short (1–2 sentences)
- Do NOT generate full explanations
- If nothing relevant, return "NONE"
- Don't return "NONE" unless absolutely nothing relevant exist

Text:
{doc.page_content}

Question:
{question}

Relevant Information:
"""

        response = llm(prompt)
        summary = response[0]['generated_text'].replace(prompt, "").strip()

        if summary != "NONE":
            summaries.append(summary)

    return summaries

In [29]:
def reduce_step(summaries, question):
    combined = "\n".join(summaries)

    prompt = f"""You are a financial analyst.

From the information below, generate a structured final answer.

Rules:
- Provide clear bullet points
- Each bullet must be UNIQUE
- Avoid repetition
- Do NOT repeat the question
- Maximum 6 bullet points

Information:
{combined}

Question:
{question}

Final Answer:
- 
"""

    response = llm(prompt)
    return response[0]['generated_text'].replace(prompt, "").strip()

In [30]:
def rag_query_advanced(question):
    docs = retriever.invoke(question)

    # MAP
    summaries = map_step(docs, question)

    # REDUCE
    final_answer = reduce_step(summaries, question)

    return final_answer, docs

In [31]:
question = "What risks does Alphabet face and how do they mitigate them?"

answer, docs = rag_query_advanced(question)

print("\n--- ANSWER ---\n")
print(answer)

print("\n--- SOURCES ---\n")
for doc in docs:
    print(doc.metadata)


--- ANSWER ---

retaliatory attack a number of factors, including, among others, reputational issues, third-party content shared on our platforms, data privacy and security issues and developments, issues in delivering age-appropriate experiences to minors, and product or technical performance failures

--- SOURCES ---

{'producer': 'Skia/PDF m146', 'creationdate': '2026-04-10T07:57:28+00:00', 'source': 'alphabet_10K.pdf', 'total_pages': 133, 'page': 12, 'moddate': '2026-04-10T07:57:28+00:00', 'title': 'goog-20251231', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36', 'page_label': '13', 'chunk_id': 49}
{'page': 25, 'source': 'alphabet_10K.pdf', 'title': 'goog-20251231', 'producer': 'Skia/PDF m146', 'chunk_id': 108, 'creationdate': '2026-04-10T07:57:28+00:00', 'moddate': '2026-04-10T07:57:28+00:00', 'total_pages': 133, 'page_label': '26', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36

## Neo4j Graph Database

##### Verifying connection the Database

In [3]:
# URI = "bolt://localhost:7687"
# USERNAME = "neo4j"
# PASSWORD = "multi-agent-assistant"

URI = "neo4j+s://f7119e85.databases.neo4j.io"
USERNAME = "f7119e85"
PASSWORD = "wjUIUGpE1D81-uZdKNxTMVs22n2Ea4kX8viuRmrrGFE"

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

with driver.session() as session:
    result = session.run("MATCH (n) RETURN count(n) AS count")
    count = result.single()["count"]
    print("Connection successful")
    print("Node count:", count)

Connection successful
Node count: 39


#### Creating a function to enter data in the Neo4j Database

In [32]:
def insert_company_data(data):
    with driver.session() as session:

        # Insert company if present
        if "company" in data and data["company"]:
            session.run(
                """
                MERGE (c:Company {name: $company})
                """,
                company = data["company"]
            )

        # Insert risks if present
        if "risks" in data and data["risks"]:
            session.run(
                """
                MATCH (c:Company {name: $company})
                UNWIND $risks AS risk
                MERGE (r:Risk {name: risk})
                MERGE (c)-[:HAS_RISK]->(r)
                """,
                company = data["company"],
                risks = data["risks"]
            )

        # Insert mitigation if present
        if "mitigation" in data and data["mitigation"]:
            session.run(
                """
                MATCH (c:Company {name: $company})
                UNWIND $mitigation AS m
                MERGE (mit:Mitigation {name: m})
                MERGE (c)-[:MITIGATES_WITH]->(mit)
                """,
                company = data["company"],
                mitigation = data["mitigation"]
            )

        # Sector (optional)
        if "sector" in data and data["sector"]:
            session.run(
                """
                MATCH (c:Company {name: $company})
                MERGE (s:Sector {name: $sector})
                MERGE (c)-[:OPERATES_IN]->(s)
                """,
                company = data["company"],
                sector = data["sector"]
            )

#### Using LLM to extract data and send that to the Neo4j Database

In [33]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3"

def extract_entities(text):
    prompt = f"""
You are an expert financial analyst.

Task:
1. Determine if this text contains information about business risks.
2. If YES, extract:
   - company (if mentioned)
   - risks (clear descriptions)
   - mitigation (if mentioned)
   
3. Ignore references to sections, items, or document structure.

5. If NO, return exactly: {{}}

Rules:
- Do NOT guess
- - Do NOT use placeholders like "..." or "N/A" or "" or "Not Explicitly Mentioned" or something similar
- Only extract clearly stated information
- Be concise and accurate

Output format (only include fields that exist):
{{
  "company": "Actual company name",
  "risks": ["Real risk 1", "Real risk 2"],
  "mitigation": ["Real mitigation 1", "Real mitigation 2"]
}}

Text:
{text}
"""
    
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
            "temperature": 0  # Lower Randomness
        }
        }
    )

    output = response.json()["response"]

    try:
        # Extract JSON block
        match = re.search(r"\{[\s\S]*?\}", output)

        if not match:
            return None

        json_str = match.group()

        return json.loads(json_str)

    except Exception as e:
        print("Parsing error:", e)
        print("Raw output:\n", output)
        return None

In [34]:
def normalize_data(data):
    ticker_map = {
                "alphabet inc.": "Alphabet",
                "alphabet inc": "Alphabet",
                "Alphabet Inc.": "Alphabet",
                "Alphabet Inc": "Alphabet",
                "GOOGL": "Alphabet",
                "GOOGLE": "Alphabet",
                "Google": "Alphabet",
                "goog": "Alphabet",
                "googl": "Alphabet",
                "google (goog)": "Alphabet",
                "goog (google)": "Alphabet",
                "Google Services": "Alphabet",
                "Alphabet (Waymo, Isomorphic Labs Are Subsidiaries)": "Alphabet",
                "Waymo": "Alphabet",
                "AAPL": "Apple",
                "MSFT": "Microsoft"
        }

    # Normalize company
    if "company" in data:
        company = data.get("company")
    
        if isinstance(company, list):
            if len(company) > 0:
                company = company[0]
            else:
                company = None
        
        if isinstance(company, str):
            if company in ticker_map:
                data["company"] = ticker_map[company]
            else:
                data["company"] = data["company"].title()
        else:
            data["company"] = None

    # Normalize sector
    if "sector" in data:
        data["sector"] = data["sector"].title()

    # Normalize risks
    normalized_risks = []
    if "risks" in data:
        for r in data["risks"]:
            r = r.lower()
    
            if "regulatory" in r:
                normalized_risks.append("Regulatory Risk")
            elif "supply" in r:
                normalized_risks.append("Supply Chain Disruption")
            elif "competition" in r:
                normalized_risks.append("Market Competition")
            else:
                normalized_risks.append(r.title())

    data["risks"] = list(set(normalized_risks))  # removing duplicates

    return data

In [35]:
def get_company_from_metadata(chunk):
    source = chunk.metadata.get("source", "").lower()

    if "alphabet" in source:
        return "Alphabet"
    elif "amazon" in source:
        return "Amazon"
    elif "apple" in source:
        return "Apple"
    elif "tesla" in source:
        return "Tesla"
    else:
        return "Microsoft"
    
    return None

In [36]:
def process_chunk(chunk, idx):
    text = chunk.page_content

    print(f"\nProcessing chunk {idx}...\n")

    data = extract_entities(text)

    if not data:
        print("Not relevant\n")
        return

    # Ensuring company exists
    if not data.get("company"):
        data["company"] = get_company_from_metadata(chunk)

    if data:
        data = normalize_data(data)
        print(data)
        insert_company_data(data)
        print("Inserted\n")
    else:
        print("No relevant information found\n")

def process_chunks_in_batches(chunks, batch_size = 50):
    # total = len(chunks)
    total = 50 # For Testing

    for i in range(0, total, batch_size):
        batch = chunks[i:i+batch_size]

        print(f"\nProcessing batch {i} to {i+batch_size-1}\n")

        for j, chunk in enumerate(batch):
            process_chunk(chunk, i + j)
            time.sleep(1)   # Prevents overload

        print("Batch complete\n")

In [37]:
process_chunks_in_batches(chunks, batch_size = 50)


Processing batch 0 to 49


Processing chunk 0...

Not relevant


Processing chunk 1...

Not relevant


Processing chunk 2...

Not relevant


Processing chunk 3...

Not relevant


Processing chunk 4...

Not relevant


Processing chunk 5...

Not relevant


Processing chunk 6...

Not relevant


Processing chunk 7...

Not relevant


Processing chunk 8...

Not relevant


Processing chunk 9...

{'company': 'Alphabet', 'risks': ['Failure To Develop New Products And Services', 'Cybersecurity Risks', 'Dependence On A Limited Number Of Customers', 'Market Competition'], 'mitigation': []}
Inserted


Processing chunk 10...

Parsing error: Expecting ',' delimiter: line 3 column 3 (char 24)
Raw output:
 Based on the provided text, I can determine that:

**YES**, this text contains information about business risks.

Here are the extracted details:

{
  "company": "GOOG"
  "risks": [],
  "mitigation": []
}

Note: There is no explicit mention of specific business risks or mitigation strategies in this

In [64]:
def link_risk_mitigation(company):
    with driver.session() as session:
        data = session.run("""
        MATCH (c:Company {name: $company})
        OPTIONAL MATCH (c)-[:HAS_RISK]->(r:Risk)
        OPTIONAL MATCH (c)-[:MITIGATES_WITH]->(m:Mitigation)
        
        RETURN collect(DISTINCT r.name) AS risks,
               collect(DISTINCT m.name) AS mitigations
        """, company=company).single()

    risks = data["risks"]
    mitigations = data["mitigations"]

    for risk in risks:
        for mitigation in mitigations:

            # simple keyword overlap logic
            if any(word in mitigation.lower() for word in risk.lower().split()):
                with driver.session() as session:
                    session.run("""
                    MATCH (r:Risk {name: $risk})
                    MATCH (m:Mitigation {name: $mitigation})
                    MERGE (r)-[:MITIGATED_BY]->(m)
                    """, risk=risk, mitigation=mitigation)

In [65]:
companies = ["Alphabet","Apple","Amazon","Microsoft","Tesla"]

for i in companies:
    link_risk_mitigation(i)

## Hybrid Retrival System <I>(Chroma Vector DB + Neo4j Graph DB)</I>

#### Building Neo4j Query Functions
- Graph Retrieval Function

In [37]:
def extract_company(query, context=None, llm=None):
    if llm:
        prompt = f"""
    Extract the company name from the query.
    
    Query: {query}
    Context: {context}
    
    Return ONLY the company name.
    If not found, return: None
    """
    
        response = call_llm(prompt)
        company = response.strip()
    
        return company if company != "None" else None
    else:
        companies = ["Alphabet", "Google", "Microsoft", "Tesla", "Apple", "Amazon"]
        
        for c in companies:
            if c.lower() in query.lower():
                # normalize Google → Alphabet
                if "google" in c.lower():
                    return "Alphabet"
                return c

In [38]:
def get_company_data(company_name):
    with driver.session() as session:
        result = session.run("""
        MATCH (c:Company {name: $company})
        OPTIONAL MATCH (c)-[:HAS_RISK]->(r:Risk)
        OPTIONAL MATCH (c)-[:MITIGATES_WITH]->(m:Mitigation)
        
        RETURN collect(DISTINCT r.name) AS risks,
               collect(DISTINCT m.name) AS mitigations
        """, company=company_name)

        data = result.single()
        return data["risks"], data["mitigations"]

#### Cleaning, Grouping and Linking the Data fetched from Neo4j

In [3]:
def filter_by_query(items, query, threshold=0.4):
    if not items:
        return []

    query_emb = embedding_model.encode([query])
    item_embs = embedding_model.encode(items)

    sims = cosine_similarity(query_emb, item_embs)[0]

    filtered = [
        item for item, sim in zip(items, sims)
        if sim > threshold
    ]

    return filtered

In [40]:
def clean_text(text):
    if not text:
        return None

    text = text.strip().lower()

    # Remove junk phrases
    junk_patterns = [
        "not explicitly mentioned",
        "none explicitly mentioned",
        "those discussed",
        "part i",
        "item",
        "see",
    ]

    for pattern in junk_patterns:
        if pattern in text:
            return None

    # Remove very long noisy sentences
    if len(text.split()) > 20:
        return None

    # Remove special chars
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)

    return text.strip()

In [41]:
def clean_list(items):
    cleaned = []

    for item in items:
        text = clean_text(item)
        if text:
            cleaned.append(text)

    return list(set(cleaned))  # remove duplicates

In [42]:
def deduplicate(items, embedding_model, threshold=0.85):
    if not items:
        return []

    embeddings = embedding_model.encode(items)
    unique_items = []

    for i, item in enumerate(items):
        is_duplicate = False

        for j, u_item in enumerate(unique_items):
            sim = cosine_similarity(
                [embeddings[i]],
                [embedding_model.encode(u_item)]
            )[0][0]

            if sim > threshold:
                is_duplicate = True
                break

        if not is_duplicate:
            unique_items.append(item)

    return unique_items

In [43]:
def group_items(items, embedding_model, similarity_threshold=0.75):
    if len(items) <= 1:
        return {items[0]: items} if items else {}

    embeddings = embedding_model.encode(items)

    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=1 - similarity_threshold,
        metric='cosine',
        linkage='average'
    )

    labels = clustering.fit_predict(embeddings)

    grouped = {}

    for label, item in zip(labels, items):
        if label not in grouped:
            grouped[label] = []
        grouped[label].append(item)

    # Convert to readable format
    final_groups = {}

    for group in grouped.values():
        representative = group[0]  # pick first as label
        final_groups[representative] = group

    return final_groups

In [44]:
def process_items(items):
    cleaned = clean_list(items)
    deduped = deduplicate(cleaned, embedding_model)
    grouped = group_items(deduped, embedding_model)
    return grouped

In [92]:
def link_risks_to_mitigations(risks, mitigations, threshold=0.3):
    if not risks or not mitigations:
        return {}

    risk_embs = embedding_model.encode(risks)
    mit_embs = embedding_model.encode(mitigations)

    links = {}

    for i, risk in enumerate(risks):
        similarities = cosine_similarity(
            [risk_embs[i]],
            mit_embs
        )[0]

        matched = [
            mitigations[j]
            for j, sim in enumerate(similarities)
            if sim > threshold
        ]

        # Explicit fallback (NO hallucination later)
        if not matched:
            matched = ["No direct mitigation found"]

        links[risk] = matched

    return links

In [93]:
def link_grouped_risks(risk_groups, mitigation_list):

    final_links = {}

    for group_name, risk_list in risk_groups.items():
        links = link_risks_to_mitigations(risk_list, mitigation_list)
        final_links[group_name] = links

    return final_links

#### Building Chroma DB Retrieval Function
- Vector DB Retrieval Function

In [94]:
def retrieve_chunks(query, k=3):
    results = db.similarity_search(query, k=k)
    return [doc.page_content for doc in results]

#### Hybrid Retrieval Function 
- Passing information from both sources to the LLM

In [95]:
def hybrid_rag_pipeline(query):

    # Retrieving chunks
    chunks = retrieve_chunks(query)
    context = "\n".join(chunks)

    # Extracting company
    company = extract_company(query, context, llm)

    if not company:
        return "Company not found in query."

    # Getting graph data
    risks, mitigations = get_company_data(company)

    # Filtering by query
    risks = filter_by_query(risks, query)
    mitigations = filter_by_query(mitigations, query)

    # Processing
    risk_groups = process_items(risks)
    mitigation_groups = process_items(mitigations)

    # Linking Risk to Mitigation
    linked_data = link_grouped_risks(
        risk_groups,
        mitigations
    )
    
    # Generating answer
    answer = generate_answer(
        query,
        company,
        context,
        linked_data
    )

    return answer

#### Building the LLM prompt and LLM

In [96]:
# OLLAMA_URL = "http://localhost:11434/api/generate"
# MODEL = "llama3"

def call_llm(prompt):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0   # deterministic output
            }
        }
    )

    return response.json()["response"]

In [97]:
def generate_answer(query, company, context, linked_data):

    structured_text = ""

    for group, risks in linked_data.items():
        structured_text += f"\n### {group}\n"
        for risk, mits in risks.items():
            structured_text += f"- Risk: {risk}\n"
            structured_text += f"  Mitigation: {', '.join(mits) if mits else 'Not specified'}\n"

    prompt = f"""
You are a financial analyst.

Query:
{query}

Company:
{company}

Context:
{context}

Structured Risk-Mitigation Mapping:
{structured_text}

Task:
- Explain risks clearly
- For EACH risk, explain its mitigation
- Do NOT generalize
- Keep mapping accurate
"""

    return call_llm(prompt)

#### Passing Query or Question to the LLM

In [79]:
query = "What cybersecurity risks does Alphabet face and how do they mitigate them?"

result = hybrid_rag_pipeline(query)
print(result)

As a financial analyst, I'll provide a structured risk-mitigation mapping for Alphabet Inc. regarding cybersecurity risks.

**Risk 1: Software Supply Chain and Third-Party Dependencies**

* Risk description: Alphabet's software supply chain and third-party dependencies pose a risk to the company's cybersecurity.
* Mitigation:
	+ Conduct thorough due diligence on third-party vendors and suppliers.
	+ Implement robust contract terms and monitoring processes for third-party dependencies.
	+ Develop and maintain a comprehensive inventory of all software components, including open-source code.

**Risk 2: Vulnerabilities in Products and Services**

* Risk description: Alphabet's products and services may contain vulnerabilities that can be exploited by attackers.
* Mitigation:
	+ Implement a robust vulnerability management program to identify, prioritize, and remediate vulnerabilities.
	+ Conduct regular security testing and penetration testing on products and services.
	+ Develop and mainta

## Multi-Agent System

#### Updating State

In [80]:
class GraphState(TypedDict):
    question: str
    
    # planner outputs
    decision: str
    reasoning: str
    
    # agent outputs
    rag_context: str
    graph_context: str
    
    # final
    answer: str

#### Planner Agent

In [81]:
def safe_parse_json(text):
    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except:
        pass
    return None

In [82]:
def planner_agent(state):
    question = state["question"]

    prompt = f"""
You are a routing agent.

Decide how to answer the question.

Options:
- rag_only
- graph_only
- hybrid

Return ONLY valid JSON.
DO NOT write anything else.

Example:
{{
  "decision": "hybrid",
  "reason": "The question asks for both structured risks and explanation"
}}

Question:
{question}
"""

    response = call_llm(prompt)

    parsed = safe_parse_json(response)

    if parsed:
        decision = parsed.get("decision", "hybrid")
        reasoning = parsed.get("reason", "")
    else:
        decision = "hybrid"
        reasoning = "Fallback due to parsing error"

    return {
        "decision": decision,
        "reasoning": reasoning
    }

#### RAG Agent

In [83]:
def rag_agent(state):
    if state["decision"] not in ["rag_only", "hybrid"]:
        return {"rag_context": ""}

    question = state["question"]

    answer, docs = rag_query(question)

    context = "\n".join([doc.page_content for doc in docs])

    return {
        "rag_context": context
    }

#### Graph Agent

In [84]:
def get_structured_graph_data(company):
    risks, mitigations = get_company_data(company)

    risk_map = link_risks_to_mitigations(risks, mitigations)

    return risk_map

In [85]:
def graph_agent(state):
    if state["decision"] not in ["graph_only", "hybrid"]:
        return {"graph_context": ""}

    question = state["question"]

    company = extract_company(question)

    if not company:
        return {"graph_context": ""}

    risk_map = get_structured_graph_data(company)

    # Convert to structured text
    formatted = ""
    for risk, mitigations in risk_map.items():
        formatted += f"\nRisk: {risk}\n"
        for m in mitigations:
            formatted += f"  - Mitigation: {m}\n"

    return {
        "graph_context": f"Company: {company}\n{formatted}"
    }

#### Final Agent <I>(Fusion)</I>

In [4]:
def final_agent(state):
    question = state["question"]

    rag_context = state.get("rag_context", "")
    graph_context = state.get("graph_context", "")
    reasoning = state.get("reasoning", "")

    prompt = f"""
You are a financial analyst.

Use ONLY the provided data.

STRICT RULES:
1. Do NOT invent mitigations
2. If mitigation is "No direct mitigation found" then:
    - infer a reasonable mitigation from available context
    - or say: "No explicit mitigation mentioned, but likely addressed through..."
    - NEVER say "No mitigation found"
3. Do NOT generalize or guess
4. Keep answer structured and precise
5. Use phrases directly grounded in context

Your job:
1. Group risks into high-level categories
2. Use professional financial categories like:
    - Cybersecurity Risk
    - Regulatory Risk
    - Advertising Revenue Risk
    - Product & Innovation Risk
    - Market Competition Risk
3. For each category:
   - summarize the risk
   - list key mitigations
4. DO NOT list raw risks
5. DO NOT say "no mitigation found"
6. Be concise and structured
7. Avoid vague terms like "technology", "brand" unless explicitly mentioned
8. Each summary MUST include specific details from context (e.g., advertising dependency, data privacy laws, cyber attacks)
9. Avoid generic phrases like "risks related to..." or "issues with..."
10. Use concrete business language
11. Avoid repeating the same mitigation across multiple categories unless explicitly stated in context
12. If mitigation is not clearly present, infer carefully or explain absence
13. Each category should have distinct, context-specific mitigations.

Planner Reasoning:
{reasoning}

Graph Data:
{graph_context}

Additional Context:
{rag_context}

Question:
{question}

Answer:
"""

    response = call_llm(prompt)

    return {"answer": response}

#### Building Graph

In [104]:
builder = StateGraph(GraphState)

builder.add_node("planner", planner_agent)
builder.add_node("rag", rag_agent)
builder.add_node("graph", graph_agent)
builder.add_node("final", final_agent)

#### Defining Flow

In [105]:
# ENTRY
builder.set_entry_point("planner")

# FLOW
builder.add_edge("planner", "rag")
builder.add_edge("planner", "graph")

builder.add_edge("rag", "final")
builder.add_edge("graph", "final")

#### Compiling

In [106]:
app = builder.compile()

#### Testing

In [107]:
result = app.invoke({
    "question": "What risks does Alphabet face and how do they mitigate them?"
})

print("Decision:", result["decision"])
print("Reason:", result["reasoning"])
print("Answer:\n", result["answer"])

Decision: hybrid
Reason: The question asks for both structured risks and explanation
Answer:
 Based on the provided data, I have grouped the risks into high-level categories and summarized each category with key mitigations. Here is the structured answer:

**Cybersecurity Risk**

* Summary: Alphabet faces cybersecurity threats, including potential attacks from nation-states and state-sponsored actors.
* Mitigation: Continuously investing in building secure products, AI-powered threat intelligence and cybersecurity solutions.

**Regulatory Risk**

* Summary: Alphabet may face regulatory risks due to changes in tax rates, adoption of new US or international tax legislation, or exposure to additional tax liabilities. Additionally, there is a risk of legal and regulatory investigations and enforcement actions.
* Mitigation: Upholding responsible data practices, enhancing these efforts over time.

**Advertising Revenue Risk**

* Summary: Alphabet generates significant revenue from advertisi

## Saving answer to some questions for Evaluation

In [117]:
questions = [
    "What risks does Alphabet face?",
    "How does Alphabet mitigate cybersecurity risks?",
    "What are the major regulatory risks for Alphabet?",
    "What risks affect Alphabet's advertising business?"
]

eval_data = []

for q in questions:
    # RAG only
    rag_answer, rag_docs = rag_query(q)

    # Hybrid (your multi-agent)
    hybrid_result = app.invoke({"question": q})
    hybrid_answer = hybrid_result["answer"]

    eval_data.append({
        "question": q,
        "rag_answer": rag_answer,
        "hybrid_answer": hybrid_answer,
        "contexts": [doc.page_content for doc in rag_docs],
        "ground_truth": ""  # we’ll fill later
    })

# Saving it
with open("eval_dataset.json", "w") as f:
    json.dump(eval_data, f, indent=2)

## Closing the Neo4j Connection Session

In [85]:
import atexit
atexit.register(lambda: driver.close())

<function __main__.<lambda>()>

# THE END